# 28 — Ablation Scaffolding

**Now (no Gemini):** retrieval ablations  
**Later (quota reset):** LLM ablations listed at the bottom

This notebook measures how retrieval settings affect **facet coverage** on the eval subset.

In [ ]:
import os
import sys
import json

project_root = os.path.abspath("..")

if project_root not in sys.path:
    sys.path.insert(0, project_root)

In [ ]:
from rag_icot.components.retriever import Retriever
from rag_icot.evaluation import (
    load_eval_dataset,
    RETRIEVAL_ABLATIONS,
    LLM_ABLATIONS,
    run_retrieval_ablation,
    summarize_ablation_table,
)

In [ ]:
print("Retrieval ablations (run now):")
for cfg in RETRIEVAL_ABLATIONS:
    print(f"- {cfg.id}: {cfg.name} | {cfg.description}")

print("\nLLM ablations (run after Gemini quota resets):")
for cfg in LLM_ABLATIONS:
    print(f"- {cfg.id}: {cfg.name} | {cfg.description}")

In [ ]:
dataset_path = os.path.join(
    project_root,
    "datasets",
    "evaluation",
    "iot_security_eval_v1.json",
)

questions = load_eval_dataset(dataset_path)

subset_ids = [
    "q001", "q002", "q011", "q019", "q021",
    "q025", "q031", "q033", "q041", "q050",
]
subset = [q for q in questions if q.id in subset_ids]
print("Subset size:", len(subset))

In [ ]:
retriever = Retriever()
rows = []

for q in subset:
    for cfg in RETRIEVAL_ABLATIONS:
        row = run_retrieval_ablation(
            q.question,
            q.required_facets,
            cfg,
            retriever=retriever,
        )
        row["question_id"] = q.id
        row["category"] = q.category
        rows.append(row)

print("Total ablation rows:", len(rows))

In [ ]:
summary = summarize_ablation_table(rows)

print(f"{'Ablation':<28} {'N':>4} {'Mean facet recall':>18}")
print("-" * 54)
for item in summary:
    print(
        f"{item['ablation_name']:<28} "
        f"{item['n']:>4} "
        f"{item['mean_facet_recall']:>18.3f}"
    )

In [ ]:
# Per-question view for the default single-pass setting
baseline_rows = [r for r in rows if r["ablation_id"] == "ret_baseline_k5"]

print(f"{'ID':<6} {'Recall':>7} {'Required':<35} {'Covered'}")
print("-" * 80)
for r in baseline_rows:
    print(
        f"{r['question_id']:<6} "
        f"{r['facet_recall']:>7.2f} "
        f"{str(r['required_facets']):<35} "
        f"{r['covered_facets']}"
    )

In [ ]:
out_dir = os.path.join(project_root, "artifacts", "evaluation")
os.makedirs(out_dir, exist_ok=True)

out_path = os.path.join(out_dir, "retrieval_ablations.json")

with open(out_path, "w", encoding="utf-8") as f:
    json.dump(
        {
            "summary": summary,
            "rows": rows,
            "llm_ablations_pending": [
                {
                    "id": cfg.id,
                    "name": cfg.name,
                    "description": cfg.description,
                    "params": cfg.params,
                }
                for cfg in LLM_ABLATIONS
            ],
        },
        f,
        indent=2,
        ensure_ascii=False,
    )

print("Saved", out_path)

## Later: LLM ablations (notebook 26 / 29)

When Gemini quota resets, compare:

1. `llm_vanilla`
2. `llm_icot_iter1`
3. `llm_icot_iter3`

Expected paper story:
- single-pass retrieval often misses facets on multi-facet questions
- source/facet routing changes coverage
- iterative ICOT improves facet recall / answer completeness vs vanilla